# Monitoring Mode

Monitoring mode is a behavioral pattern where the agent passively observes a system, data stream, or environment and takes action only when predefined conditions are met. Unlike conversational or task-execution modes that respond turn-by-turn to user input, a monitoring agent runs continuously in the background and produces output only when something requires attention.

The core principle: the agent is quiet unless something is wrong. An agent that sends frequent reports when everything is normal creates alert fatigue — humans learn to ignore its output. An agent that fires only on genuine conditions earns trust.

Monitoring mode is classified as passive observation, alert-on-condition. It can combine with any autonomy level:
- Monitoring + copilot: Agent detects conditions and alerts humans who then act.
- Monitoring + supervised: Agent takes initial response actions and reports for confirmation.
- Monitoring + fully autonomous: Agent detects and remediates conditions automatically.

In [1]:
import os
import json
import time
import random
from dataclasses import dataclass
from typing import Callable, Optional
from enum import Enum

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

### Initialize the language model

Monitoring mode uses `temperature=0` for deterministic condition evaluation. When the same observation is checked against the same condition twice, the verdict must be identical both times — non-deterministic evaluation would cause the same event to trigger an alert in one cycle and not the next, making the monitoring system unreliable.

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY", "").strip())

print("LLM initialized for monitoring mode.")

LLM initialized for monitoring mode.


## Defining the monitoring data model

The data model for monitoring mode has three layers: the conditions being watched, the alerts they generate, and the observation records that log what the agent has seen.

`MonitoringCondition` captures everything needed to define one watchpoint: a description of what to look for, the severity of the alert when it fires, a check function that evaluates current observations, the kind of response to take, and a cooldown that prevents the same condition from firing repeatedly within a short window. The `cooldown_seconds` field is as important as the condition itself — a monitoring system without cooldowns can saturate alert channels in minutes if an intermittent condition fires every cycle.

`Alert` carries the full context of a triggered event: what happened, when, the severity, the supporting data, and a recommended action. A well-formed alert is self-contained — the responder should be able to act without querying the monitoring system for more information. `ObservationRecord` logs every cycle for audit purposes, including cycles that triggered no alerts.

In [3]:
class AlertSeverity(Enum):
    INFO     = "info"
    WARNING  = "warning"
    CRITICAL = "critical"


@dataclass
class MonitoringCondition:
    """A condition that, when met, triggers an alert and optional remediation."""
    name: str                  # short identifier used in logs and alert routing
    description: str           # plain-language description of what is being watched
    severity: AlertSeverity    # how serious is this condition when triggered
    check_function: Callable   # (observations: dict) → (triggered: bool, context: dict)
    response_action: str       # 'alert_only' | 'alert_and_remediate'
    cooldown_seconds: int = 300    # minimum seconds between alerts for this condition
    last_triggered: float = 0.0   # unix timestamp of last trigger; updated by run_cycle


@dataclass
class Alert:
    """A structured alert produced when a monitoring condition is triggered."""
    condition_name: str       # which condition fired
    severity: AlertSeverity
    timestamp: float          # unix timestamp of when the alert was generated
    context: dict             # data points that caused the trigger
    message: str              # human-readable summary of what happened
    recommended_action: str   # specific steps for the responding team


@dataclass
class ObservationRecord:
    """One monitoring cycle logged for audit and forensic analysis."""
    timestamp: float
    observations: dict        # raw data collected this cycle from all sources
    conditions_checked: int   # how many conditions were evaluated (excluding cooled-down ones)
    alerts_triggered: int     # how many alerts fired this cycle


print("Monitoring data model defined.")
print(f"  AlertSeverity  : {[s.value for s in AlertSeverity]}")
print(f"  Response types : alert_only | alert_and_remediate")

Monitoring data model defined.
  AlertSeverity  : ['info', 'warning', 'critical']
  Response types : alert_only | alert_and_remediate


`MonitoringCondition.check_function` is typed as `Callable` rather than as a fixed signature. This allows any callable — a plain function, a lambda, or later a closure — to serve as the check. The contract is informal: the function receives an observations dict and returns `(triggered: bool, context: dict)`. Keeping it as a `Callable` rather than a strict protocol means conditions can be defined as module-level functions, closures over thresholds, or even LLM-backed callables without any change to the monitoring loop.

`last_triggered` defaults to `0.0`, which means every condition is eligible to fire on the very first cycle — there is no artificial warmup delay. As soon as a condition fires, `run_cycle` writes the current unix timestamp into this field. On every subsequent cycle, the elapsed time since `last_triggered` is compared against `cooldown_seconds` before the check function is even called.

## Defining monitoring conditions
Monitoring conditions are the core configuration of a monitoring agent. Each condition is a function that receives the current observations and returns two things: whether the condition is triggered, and the context data that explains why. Well-designed check functions have two properties: they are precise (fire only when the condition is actually true), and they are safe to run every cycle (no network calls, no side effects, and explicit handling of missing or unavailable data).

The two functions below watch the error rate and response latency of a simulated API service. Both begin by checking whether the data source is marked as unavailable — if it is, they return `False` immediately rather than triggering an alert based on missing data. This is the most important safety rule for check functions: when you cannot observe, do not alert.

In [4]:
def check_high_error_rate(observations: dict) -> tuple:
    """Trigger when API error rate exceeds 5%."""
    api_data = observations.get("api_metrics", {})

    if api_data.get("status") == "unavailable":
        # Data source is down — cannot evaluate; return False to avoid a false positive
        return False, {}

    error_rate = api_data.get("error_rate_pct", 0.0)
    threshold = 5.0
    triggered = error_rate > threshold   # boolean result of the condition check

    # Include the raw numbers so the alert message can cite specific evidence
    context = {
        "error_rate_pct": error_rate,
        "threshold_pct": threshold,
        "total_requests": api_data.get("total_requests", 0),
        "error_count": api_data.get("error_count", 0),
    }
    return triggered, context


def check_high_latency(observations: dict) -> tuple:
    """Trigger when p99 response latency exceeds 2000ms."""
    api_data = observations.get("api_metrics", {})

    if api_data.get("status") == "unavailable":
        return False, {}

    p99_ms = api_data.get("p99_latency_ms", 0)
    threshold_ms = 2000
    triggered = p99_ms > threshold_ms

    context = {
        "p99_latency_ms": p99_ms,
        "threshold_ms": threshold_ms,
        "p50_latency_ms": api_data.get("p50_latency_ms", 0),  # p50 helps distinguish spike from sustained
    }
    return triggered, context


# Quick sanity check with a synthetic observation
sample_obs = {"api_metrics": {"status": "ok", "error_rate_pct": 12.5, "error_count": 125,
                               "total_requests": 1000, "p99_latency_ms": 420, "p50_latency_ms": 85}}
triggered_err, ctx_err = check_high_error_rate(sample_obs)
triggered_lat, ctx_lat = check_high_latency(sample_obs)
print(f"Error rate check : triggered={triggered_err}, rate={ctx_err['error_rate_pct']}%")
print(f"Latency check    : triggered={triggered_lat}, p99={ctx_lat['p99_latency_ms']}ms")

Error rate check : triggered=True, rate=12.5%
Latency check    : triggered=False, p99=420ms


Both check functions follow the same three-step structure: guard against unavailable data, compute the threshold comparison, and return `(triggered, context)`. The guard comes first so that an unavailable data source does not produce a false positive — returning `(False, {})` means the monitoring loop will log a silent OK for that condition rather than firing an alert about missing data.

The `context` dict captures both the observed value and the threshold. Including the threshold alongside the observed value means the alert message can read "error rate is 12.5%, threshold is 5%" rather than just "error rate is 12.5%" — the responder immediately understands the magnitude of the violation without needing to look up the configuration.

The traffic spike condition introduces a ratio-based check rather than an absolute threshold. A request rate of 750 RPM is only alarming if the baseline is 200 RPM; the same rate would be normal for a system that routinely handles 600 RPM. Comparing current volume against a rolling baseline makes the condition self-calibrating — it adapts to the system's normal traffic without requiring manual threshold updates.

With all three check functions defined, we assemble the `CONDITIONS` list. Notice the different cooldown values: the error rate alert (most severe) has the shortest cooldown at 5 minutes because a sustained error rate is a live incident that should keep alerting. The traffic spike alert has the longest cooldown at 15 minutes because traffic spikes often have a natural resolution within that window.

In [5]:
def check_traffic_spike(observations: dict) -> tuple:
    """Trigger when request volume is more than 3x the hourly average."""
    api_data = observations.get("api_metrics", {})
    traffic_data = observations.get("traffic_baseline", {})

    if api_data.get("status") == "unavailable":
        return False, {}

    current_rpm = api_data.get("requests_per_minute", 0)
    # Default to 1 to avoid ZeroDivisionError if baseline is missing
    baseline_rpm = traffic_data.get("hourly_avg_rpm", 1)
    ratio = current_rpm / baseline_rpm
    triggered = ratio > 3.0   # 3x baseline is a meaningful spike, not just normal variance

    context = {
        "current_rpm": current_rpm,
        "baseline_rpm": baseline_rpm,
        "spike_ratio": round(ratio, 2),
    }
    return triggered, context


# Assemble conditions with different cooldowns reflecting the urgency of each
CONDITIONS = [
    MonitoringCondition(
        name="high_error_rate",
        description="API error rate exceeds 5% — systemic problem affecting users",
        severity=AlertSeverity.CRITICAL,
        check_function=check_high_error_rate,
        response_action="alert_and_remediate",
        cooldown_seconds=300,    # 5 min — keep re-alerting on an active incident
    ),
    MonitoringCondition(
        name="high_latency",
        description="p99 API latency exceeds 2000ms — users experiencing slow responses",
        severity=AlertSeverity.WARNING,
        check_function=check_high_latency,
        response_action="alert_only",
        cooldown_seconds=600,    # 10 min — allow time for natural resolution
    ),
    MonitoringCondition(
        name="traffic_spike",
        description="Request volume is 3x the hourly average — possible abuse or viral event",
        severity=AlertSeverity.WARNING,
        check_function=check_traffic_spike,
        response_action="alert_only",
        cooldown_seconds=900,    # 15 min — spikes often resolve on their own
    ),
]

print(f"Configured {len(CONDITIONS)} monitoring conditions:")
for cond in CONDITIONS:
    print(f"  [{cond.severity.value.upper():8}] {cond.name} (cooldown: {cond.cooldown_seconds}s)")

Configured 3 monitoring conditions:
  [CRITICAL] high_error_rate (cooldown: 300s)
  [WARNING ] high_latency (cooldown: 600s)
  [WARNING ] traffic_spike (cooldown: 900s)


`check_traffic_spike` uses `traffic_data.get("hourly_avg_rpm", 1)` with a default of 1 rather than 0. A default of 0 would produce a `ZeroDivisionError` if the baseline data source is unavailable; a default of 1 means any non-zero current volume would technically trigger the condition — which is also wrong, but at least it is handleable. In practice, the `status == "unavailable"` guard earlier in the function catches most baseline-missing cases before the division.

The `CONDITIONS` list is ordered by severity with the most critical condition first. This ordering matters in the monitoring loop: if the cycle has limited time and is interrupted partway through, the most important conditions have already been evaluated. The different `cooldown_seconds` values encode a policy decision: how long should the team have to respond before the same alert fires again.

## Simulating data sources
In production, a monitoring agent reads from real data sources — metrics APIs, log aggregators, time-series databases. For this notebook, `fetch_observations` generates synthetic data keyed by source name. The function accepts a `scenario` argument that overrides the healthy baseline with a degraded state, letting us exercise each condition without needing a real production system.

The dict-of-dicts structure (observations keyed by source, data keyed within each source) matches how a production monitoring agent would aggregate readings from multiple independent sources. Each check function reads from the key it owns and ignores the rest, so adding a new data source does not require modifying any existing check.

In [6]:
def fetch_observations(scenario: str = "normal") -> dict:
    """Simulate fetching observations from monitored data sources.

    Args:
        scenario: 'normal' | 'high_errors' | 'high_latency' | 'traffic_spike' | 'multiple'

    Returns:
        Dict of observations keyed by source name.
    """
    # Baseline values representing a healthy system under normal load
    base = {
        "api_metrics": {
            "error_rate_pct": 0.8,
            "error_count": 8,
            "total_requests": 1000,
            "p99_latency_ms": 420,
            "p50_latency_ms": 85,
            "requests_per_minute": 180,
            "status": "ok",
        },
        "traffic_baseline": {
            "hourly_avg_rpm": 200,
        },
    }

    # Each degraded scenario overrides only the relevant metrics, leaving the rest healthy
    overrides = {
        "high_errors":   {"api_metrics": {**base["api_metrics"], "error_rate_pct": 12.5, "error_count": 125}},
        "high_latency":  {"api_metrics": {**base["api_metrics"], "p99_latency_ms": 3800, "p50_latency_ms": 950}},
        "traffic_spike": {"api_metrics": {**base["api_metrics"], "requests_per_minute": 750}},
        # 'multiple' combines several degraded metrics to demonstrate multi-condition cycles
        "multiple": {"api_metrics": {**base["api_metrics"],
                                     "error_rate_pct": 8.0, "error_count": 80,
                                     "p99_latency_ms": 3200, "requests_per_minute": 720}},
    }

    if scenario in overrides:
        return {**base, **overrides[scenario]}   # merge: override-keys replace base-keys
    return base   # 'normal' or unrecognized scenario returns the healthy baseline


# Compare normal vs. degraded to confirm the simulation produces the right values
normal_obs   = fetch_observations("normal")
degraded_obs = fetch_observations("high_errors")

print("Normal observations:")
print(f"  Error rate : {normal_obs['api_metrics']['error_rate_pct']}%")
print(f"  p99 latency: {normal_obs['api_metrics']['p99_latency_ms']}ms")
print(f"  RPM        : {normal_obs['api_metrics']['requests_per_minute']}")

print("\nDegraded observations (high errors):")
print(f"  Error rate : {degraded_obs['api_metrics']['error_rate_pct']}%  ← above 5% threshold")
print(f"  p99 latency: {degraded_obs['api_metrics']['p99_latency_ms']}ms")
print(f"  RPM        : {degraded_obs['api_metrics']['requests_per_minute']}")

Normal observations:
  Error rate : 0.8%
  p99 latency: 420ms
  RPM        : 180

Degraded observations (high errors):
  Error rate : 12.5%  ← above 5% threshold
  p99 latency: 420ms
  RPM        : 180


`fetch_observations` uses the spread operator `{**base, **overrides[scenario]}` to merge the base observations with scenario overrides. Because `overrides` replaces entire source entries (e.g., the full `"api_metrics"` dict), and individual metric overrides use `{**base["api_metrics"], ...}` inline, the result is always a fully-specified observation with no missing keys — which prevents `KeyError` exceptions in any check function.

The `"multiple"` scenario is included specifically to demonstrate that the monitoring loop handles multiple simultaneous condition triggers correctly, including generating separate alerts for each and handling their individual cooldowns independently.

## Generating structured alerts
When a condition triggers, the monitoring agent must produce an alert that is useful to the responder. An alert that says only "error rate is high" requires the responder to query the system themselves to understand what happened. An alert with specific data points, context, and a recommended action lets the responder start immediately.

The LLM generates the alert message and recommended action from the condition description and the triggering context. This is one of the few places in monitoring mode where the LLM adds genuine value over a fixed template: it synthesizes the context into a natural-language summary that conveys urgency appropriate to the severity, and it produces recommendations specific to the triggering data rather than generic advice.

In [7]:
def create_alert(condition: MonitoringCondition, context: dict) -> Alert:
    """Generate a structured alert when a monitoring condition is triggered.

    Args:
        condition: The MonitoringCondition that fired.
        context: The data points returned by the check function.

    Returns:
        A fully formed Alert ready for routing to handlers.
    """
    # The system prompt enforces the JSON schema — message + recommended_action only
    system_prompt = """You are an alert system. Generate a concise, actionable alert.

Respond with JSON only:
{"message": "1-2 sentences describing what happened", "recommended_action": "specific steps for the responding team"}"""

    # Pass the condition description and context numbers so the model can be specific
    prompt = f"""Condition triggered:
  Name       : {condition.name}
  Description: {condition.description}
  Severity   : {condition.severity.value}
  Context    : {json.dumps(context, indent=2)}

Generate an alert message and recommended action. Cite the numbers from the context."""

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    try:
        alert_content = json.loads(response.content)
    except json.JSONDecodeError:
        # Strip markdown fences if the model wrapped the JSON in a code block
        raw = response.content.strip()
        if raw.startswith("```"):
            raw = "\n".join(raw.split("\n")[1:-1])
        alert_content = json.loads(raw)

    return Alert(
        condition_name=condition.name,
        severity=condition.severity,
        timestamp=time.time(),
        context=context,
        # Fall back to a minimal message if parsing somehow failed
        message=alert_content.get("message", f"{condition.name} triggered"),
        recommended_action=alert_content.get("recommended_action", "Investigate immediately."),
    )

In [8]:
# Trigger the high_error_rate check and create an alert from its output
triggered, context = check_high_error_rate(fetch_observations("high_errors"))

if triggered:
    alert = create_alert(CONDITIONS[0], context)

    print("Alert generated:")
    print("=" * 60)
    print(f"Condition : {alert.condition_name}")
    print(f"Severity  : {alert.severity.value.upper()}")
    print(f"Timestamp : {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(alert.timestamp))}")
    print(f"\nMessage:\n  {alert.message}")
    print(f"\nRecommended action:\n  {alert.recommended_action}")
    print("\nSupporting context:")
    for k, v in alert.context.items():
        print(f"  {k}: {v}")

Alert generated:
Condition : high_error_rate
Severity  : CRITICAL
Timestamp : 2026-05-10 08:46:48

Message:
  A critical alert has been triggered due to a high error rate of 12.5%, exceeding the threshold of 5%. This systemic issue is affecting users, with 125 errors out of 1000 total requests.

Recommended action:
  Investigate the root cause of the API errors immediately. Review logs and metrics to identify any recent changes or anomalies. Consider rolling back recent deployments if necessary and communicate with affected users.

Supporting context:
  error_rate_pct: 12.5
  threshold_pct: 5.0
  total_requests: 1000
  error_count: 125


`create_alert` is called only after the check function has returned `triggered=True`, so it always receives a non-empty context dict. The LLM call is deliberately isolated from the check functions — checks must be cheap and synchronous, but alert generation can afford one LLM call since it happens only when a condition fires.

The `alert_content.get("message", ...)` fallback ensures the function never raises even if the JSON response is missing a field. The fallback values are minimal but valid: a responder receiving `"high_error_rate triggered"` with no action still knows which condition fired and can look up the playbook.

## The monitoring cycle
The `run_cycle` function is the heart of monitoring mode. It takes the current observations, iterates through every configured condition, and for each one: checks the cooldown, runs the check function, generates an alert if triggered, and updates the condition's cooldown timestamp. All of this happens in one pass and returns the list of alerts generated.

Two design decisions in this loop are worth calling out. First, cooldown enforcement happens *before* calling the check function, not after. This means a condition that is still within its cooldown window is skipped entirely — no check function overhead, no false trigger risk from a malfunctioning function during cooldown. Second, each check function runs inside its own `try/except`. A bug in one check function must not crash the entire cycle — all remaining conditions must still be evaluated.

In [9]:
def run_cycle(
    conditions: list,
    observations: dict,
    alert_handlers: list,
    alert_history: list,
    observation_log: list,
) -> list:
    """Run one monitoring cycle: evaluate all conditions and fire alerts for triggered ones.

    Args:
        conditions: List of MonitoringCondition objects to evaluate.
        observations: Current observations from all monitored data sources.
        alert_handlers: Callables invoked when an alert fires (e.g., log, notify).
        alert_history: Running list of all alerts; new alerts are appended here.
        observation_log: Running list of ObservationRecord; one record added per cycle.

    Returns:
        List of Alert objects generated in this cycle.
    """
    current_time = time.time()
    cycle_alerts = []       # alerts generated in this specific cycle
    conditions_checked = 0  # count excludes cooled-down conditions

    for condition in conditions:
        # Cooldown check: skip this condition if it fired too recently
        elapsed = current_time - condition.last_triggered
        if elapsed < condition.cooldown_seconds:
            remaining = int(condition.cooldown_seconds - elapsed)
            print(f"  [COOLDOWN] {condition.name}: {remaining}s remaining")
            continue   # do not call the check function during cooldown

        conditions_checked += 1

        try:
            triggered, context = condition.check_function(observations)

            if triggered:
                print(f"  [TRIGGERED] {condition.name} ({condition.severity.value.upper()})")

                alert = create_alert(condition, context)
                alert_history.append(alert)   # permanent record across all cycles
                cycle_alerts.append(alert)    # this cycle's alerts only

                # Update cooldown timestamp only on a trigger — not on a clean check
                condition.last_triggered = current_time

                # Deliver to all registered handlers (log, notify, remediate, etc.)
                for handler in alert_handlers:
                    try:
                        handler(alert)
                    except Exception as handler_err:
                        # A broken handler must not discard remaining handlers
                        print(f"  [HANDLER ERROR] {handler_err}")
            else:
                print(f"  [OK      ] {condition.name}")

        except Exception as check_err:
            # A broken check function must not crash the entire monitoring cycle
            print(f"  [ERROR   ] {condition.name}: {check_err}")

    # Log this cycle regardless of whether any alerts fired
    observation_log.append(ObservationRecord(
        timestamp=current_time,
        observations=observations,
        conditions_checked=conditions_checked,
        alerts_triggered=len(cycle_alerts),
    ))

    return cycle_alerts

`run_cycle` accepts `alert_history` and `observation_log` as external lists rather than maintaining them internally. This makes the function stateless with respect to history — all state lives in the caller's scope and is passed in. The only state mutation that happens inside `run_cycle` is `condition.last_triggered = current_time`, which writes directly to the `MonitoringCondition` dataclass field. This in-place update is what persists the cooldown across cycle calls without needing a separate data structure.

The nested `try/except` around each handler call is separate from the `try/except` around the check function. This means a broken alert handler (e.g., a Slack notification that times out) does not prevent the condition's cooldown timestamp from being updated, and it does not prevent other handlers from receiving the same alert.

## Demonstration: three monitoring cycles
The three cycles below demonstrate the full behavior of the monitoring loop. Cycle 1 observes a healthy system — all conditions pass silently. Cycle 2 observes a degraded system with multiple simultaneous failures — conditions fire, alerts are generated, and cooldown timestamps are updated. Cycle 3 observes the same degraded state — but all conditions are now within their cooldown windows, so no alerts fire even though the conditions would still be triggered.

Cycle 3 demonstrates the "alert-on-condition, not alert-continuously" principle mechanically. The monitoring system has already notified the team; re-alerting every cycle while the incident is being handled would add noise, not information.

In [10]:
# Alert handler: prints alerts to the console (in production: Slack, PagerDuty, etc.)
def console_alert_handler(alert: Alert):
    print(f"  \u250c\u2500 ALERT [{alert.severity.value.upper()}] {alert.condition_name}")
    print(f"  \u2502  {alert.message}")
    print(f"  \u2514\u2500 Action: {alert.recommended_action[:80]}")


# Shared state passed to run_cycle each call
alert_history   = []   # all alerts ever generated
observation_log = []   # one record per cycle

print("\u2550" * 60)
print("CYCLE 1: Normal conditions")
print("\u2550" * 60)
cycle1 = run_cycle(CONDITIONS, fetch_observations("normal"), [console_alert_handler], alert_history, observation_log)
print(f"\nAlerts fired: {len(cycle1)}")

# Reset cooldowns so cycle 2 can fire (simulates time passing between cycles)
for cond in CONDITIONS:
    cond.last_triggered = 0.0

print("\n\u2550" * 2 + "\u2550" * 58)
print("CYCLE 2: Multiple degraded conditions")
print("\u2550" * 60)
cycle2 = run_cycle(CONDITIONS, fetch_observations("multiple"), [console_alert_handler], alert_history, observation_log)
print(f"\nAlerts fired: {len(cycle2)}")

print("\n\u2550" * 2 + "\u2550" * 58)
print("CYCLE 3: Same degraded conditions — cooldowns now active")
print("\u2550" * 60)
cycle3 = run_cycle(CONDITIONS, fetch_observations("multiple"), [console_alert_handler], alert_history, observation_log)
print(f"\nAlerts fired: {len(cycle3)}  (suppressed by cooldown \u2014 expected)")

print("\nMonitoring summary:")
print(f"  Cycles run   : {len(observation_log)}")
print(f"  Total alerts : {len(alert_history)}")
alert_by_sev = {s.value: sum(1 for a in alert_history if a.severity == s) for s in AlertSeverity}
for sev, count in alert_by_sev.items():
    print(f"  {sev.upper():8}: {count}")

════════════════════════════════════════════════════════════
CYCLE 1: Normal conditions
════════════════════════════════════════════════════════════
  [OK      ] high_error_rate
  [OK      ] high_latency
  [OK      ] traffic_spike

Alerts fired: 0

═
═══════════════════════════════════════════════════════════
CYCLE 2: Multiple degraded conditions
════════════════════════════════════════════════════════════
  [TRIGGERED] high_error_rate (CRITICAL)
  ┌─ ALERT [CRITICAL] high_error_rate
  │  The API error rate has exceeded the critical threshold of 5%, currently at 8% with 80 errors out of 1000 total requests. This systemic issue is affecting users significantly.
  └─ Action: Investigate the root cause of the errors immediately. Review logs for the last h
  [TRIGGERED] high_latency (WARNING)
  ┌─ ALERT [WARNING] high_latency
  │  High latency condition triggered: p99 API latency has exceeded 2000ms, currently at 3200ms, causing slow responses for users. The p50 latency is at 85ms, indicat

The cooldown reset between cycles 1 and 2 (`cond.last_triggered = 0.0`) simulates real elapsed time. In a production deployment, the monitoring loop runs on a timer (e.g., every 60 seconds), and cooldown windows naturally expire between cycles without any explicit reset. The reset here is only needed because this notebook runs all three cycles in rapid succession.

Cycle 3 produces zero alerts even though the same degraded observations are fed in. This confirms that the cooldown check fires before the condition check — the conditions would still trigger if evaluated, but the loop skips them entirely. The `observation_log` still records a cycle entry with `alerts_triggered=0`, which is important for auditing: it proves the monitoring agent ran and observed the degraded state, even though it chose not to re-alert.

## LLM-driven condition checking
The threshold-based conditions above are fast and predictable, but they cannot handle conditions that require language understanding — for example, detecting whether user feedback contains expressions of frustration, or whether a log entry describes a security event that does not match any known pattern.

The `llm_condition_check` function below uses the LLM as the check function itself. It evaluates observations against a natural-language condition description, calibrated by example triggers and non-triggers. The confidence threshold (0.8 by default) is the critical design decision: the LLM must be at least 80% confident before an alert fires. In monitoring mode, false positives are more damaging than false negatives — an alert that fires repeatedly without cause trains operators to ignore the channel, and when a real incident fires the same alert, it gets ignored too.

In [11]:
def llm_condition_check(
    observations: dict,
    condition_description: str,
    example_triggers: list,
    example_non_triggers: list,
    confidence_threshold: float = 0.8,
) -> tuple:
    """Evaluate a monitoring condition using LLM interpretation.

    Use for conditions that require language understanding and cannot be expressed
    as simple threshold comparisons.

    Args:
        observations: Current observations to evaluate.
        condition_description: Natural-language description of what should trigger.
        example_triggers: Concrete examples of situations that SHOULD trigger.
        example_non_triggers: Concrete examples that should NOT trigger.
        confidence_threshold: Minimum confidence (0–1) required to fire. Default 0.8.

    Returns:
        Tuple of (triggered: bool, context: dict with confidence and reasoning).
    """
    # System prompt instructs the model to be conservative — false positives erode trust
    system_prompt = """You are a monitoring system evaluating whether a condition is triggered.
Be conservative — only trigger if confidence is high. False positives erode trust.

Respond with JSON only:
{"triggered": true/false, "confidence": 0.0-1.0, "reason": "why or why not", "relevant_data": {...}}"""

    prompt = f"""Monitoring condition: {condition_description}

Situations that SHOULD trigger this alert:
{json.dumps(example_triggers, indent=2)}

Situations that should NOT trigger this alert:
{json.dumps(example_non_triggers, indent=2)}

Current observations:
{json.dumps(observations, indent=2)}

Is this condition triggered?"""

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    try:
        result = json.loads(response.content)
    except json.JSONDecodeError:
        raw = response.content.strip()
        if raw.startswith("```"):
            raw = "\n".join(raw.split("\n")[1:-1])
        result = json.loads(raw)

    confidence = result.get("confidence", 0.0)
    # Apply confidence gate: even if triggered=True, suppress below threshold
    triggered = result.get("triggered", False) and confidence >= confidence_threshold

    context = {
        "triggered": triggered,
        "confidence": confidence,
        "reason": result.get("reason", ""),
        "relevant_data": result.get("relevant_data", {}),
    }
    return triggered, context

In [12]:
# Shared configuration for the user frustration checker
FRUSTRATION_CONDITION = "User feedback contains strong expressions of frustration, anger, or threat to cancel"
TRIGGER_EXAMPLES = [
    "This is completely broken and I'm about to cancel my subscription",
    "I've been waiting 3 hours and nothing works. This is unacceptable.",
]
NON_TRIGGER_EXAMPLES = [
    "I'd prefer if the export was faster, but otherwise it works fine",
    "Having a small issue with the date format, can you help?",
]

# Two test cases — one should trigger, one should not
test_cases = [
    {
        "feedback": "The data export has been failing for 2 days. My whole team is blocked. Fix this or we're canceling.",
        "user_id": "usr_4421", "plan": "enterprise",
    },
    {
        "feedback": "Could you add dark mode? Would be a nice improvement.",
        "user_id": "usr_8832", "plan": "pro",
    },
]

print("LLM-driven condition checker results:")
print("=" * 60)
for case in test_cases:
    triggered, context = llm_condition_check(
        observations=case,
        condition_description=FRUSTRATION_CONDITION,
        example_triggers=TRIGGER_EXAMPLES,
        example_non_triggers=NON_TRIGGER_EXAMPLES,
        confidence_threshold=0.8,
    )
    print(f"Feedback  : \"{case['feedback'][:65]}...\"")
    print(f"  Triggered : {triggered}")
    print(f"  Confidence: {context['confidence']}")
    print(f"  Reason    : {context['reason']}")
    print()

LLM-driven condition checker results:
Feedback  : "The data export has been failing for 2 days. My whole team is blo..."
  Triggered : True
  Confidence: 0.9
  Reason    : The feedback contains strong expressions of frustration and a clear threat to cancel the subscription if the issue is not resolved.

Feedback  : "Could you add dark mode? Would be a nice improvement...."
  Triggered : False
  Confidence: 0.9
  Reason    : The feedback does not express strong frustration, anger, or a threat to cancel. It is a suggestion for improvement.



`llm_condition_check` applies the confidence gate as `result.get("triggered", False) and confidence >= confidence_threshold`. This means a trigger is only possible if the model explicitly returns `triggered: true` AND the confidence meets or exceeds the threshold. A model that returns `triggered: true` with confidence `0.65` will be treated as not triggered — the boolean and float operate as a joint gate.

The `example_triggers` and `example_non_triggers` arguments act as few-shot examples in the prompt. Including non-trigger examples is as important as including trigger examples: without them, the model may classify ambiguous inputs (like a mildly frustrated but non-threatening complaint) as triggers. The non-trigger examples calibrate the model's sensitivity, serving the same purpose as labeled negative examples in a classifier.

## Calibrating alert thresholds
Setting alert thresholds is one of the most consequential decisions in monitoring mode. A threshold set too low produces frequent false alarms that train operators to ignore the channel; a threshold set too high misses genuine problems until they become severe. Neither extreme is acceptable in production.

The function below uses historical data to compute statistics-based thresholds. The standard deviation approach — setting thresholds at N standard deviations above the mean — is a principled alternative to picking numbers by intuition. At 2.5 standard deviations, approximately 1.2% of normal observations would exceed the threshold. This rate is low enough to avoid alert fatigue, while being sensitive enough to catch genuine anomalies that static thresholds at round numbers might miss.

In [13]:
def calibrate_threshold(historical_values: list) -> dict:
    """Compute alert thresholds from historical data using standard deviation analysis.

    Args:
        historical_values: Metric values observed during normal (non-incident) operation.

    Returns:
        Dict with mean, std_dev, and threshold options at 2, 2.5, and 3 standard deviations.
    """
    if len(historical_values) < 10:
        raise ValueError("Need at least 10 historical data points for reliable calibration.")

    n = len(historical_values)
    mean = sum(historical_values) / n

    # Population variance: sum of squared deviations from mean, divided by N
    variance = sum((x - mean) ** 2 for x in historical_values) / n
    std_dev = variance ** 0.5   # square root of variance gives standard deviation

    return {
        "sample_count": n,
        "mean": round(mean, 2),
        "std_dev": round(std_dev, 2),
        # Thresholds at increasing sigma levels — higher sigma = fewer false positives
        "threshold_2_sigma":   round(mean + 2   * std_dev, 2),   # ~2.3% false positive rate
        "threshold_2_5_sigma": round(mean + 2.5 * std_dev, 2),   # ~1.2% false positive rate
        "threshold_3_sigma":   round(mean + 3   * std_dev, 2),   # ~0.1% false positive rate
        "recommended": round(mean + 2.5 * std_dev, 2),           # 2.5σ is a practical default
    }


# Simulate 30 days of hourly error rate readings during normal operation (720 samples)
random.seed(42)
historical = [random.gauss(mu=1.2, sigma=0.6) for _ in range(720)]
historical = [max(0.0, v) for v in historical]   # error rates cannot be negative

cal = calibrate_threshold(historical)

print("Error rate threshold calibration (30 days of historical data):")
print("=" * 60)
print(f"Sample count     : {cal['sample_count']} observations")
print(f"Baseline mean    : {cal['mean']}%")
print(f"Std deviation    : {cal['std_dev']}%")
print()
print(f"Threshold at 2\u03c3   : {cal['threshold_2_sigma']}%   (~2.3% false positives)")
print(f"Threshold at 2.5\u03c3 : {cal['threshold_2_5_sigma']}%  (~1.2% false positives)  \u2190 recommended")
print(f"Threshold at 3\u03c3   : {cal['threshold_3_sigma']}%   (~0.1% false positives)")
print()
print(f"Current hard threshold : 5.0%  (set manually in CONDITIONS)")
print(f"Data-driven suggestion : {cal['recommended']}%")

Error rate threshold calibration (30 days of historical data):
Sample count     : 720 observations
Baseline mean    : 1.2%
Std deviation    : 0.58%

Threshold at 2σ   : 2.36%   (~2.3% false positives)
Threshold at 2.5σ : 2.65%  (~1.2% false positives)  ← recommended
Threshold at 3σ   : 2.94%   (~0.1% false positives)

Current hard threshold : 5.0%  (set manually in CONDITIONS)
Data-driven suggestion : 2.65%


`calibrate_threshold` computes population variance (dividing by `n`) rather than sample variance (dividing by `n-1`). For 720 data points the difference is negligible, and population variance is appropriate when the historical data represents the full distribution of normal behavior rather than a random sample from a larger population.

The 2.5σ recommendation balances sensitivity against false positive rate. A 3σ threshold is often too conservative for monitoring: at 0.1% false positives with a cycle every 60 seconds, the system would still produce roughly one false alert per day. The output comparison between the manually set 5.0% threshold and the data-driven suggestion illustrates why calibration matters — intuitive round numbers frequently differ from what the historical distribution actually supports.

- **Monitoring mode is exception-driven, not report-driven**: The agent produces output only when a condition triggers. An agent that sends frequent reports during normal operation creates alert fatigue and trains operators to ignore it.
- **Cooldowns are as important as conditions**: Without cooldowns, a persistent condition fires every cycle, saturating alert channels and producing the same fatigue as over-reporting. Cooldowns enforce the alert-on-condition principle mechanically.
- **False positives are more damaging than false negatives in most monitoring contexts**: An alert that fires repeatedly without cause trains operators to dismiss it — which means when a genuine problem fires the same alert, it gets ignored. Conservative thresholds and high confidence requirements protect the system's credibility.
- **LLM-driven checking extends monitoring to nuanced conditions**: Simple thresholds cannot detect user frustration, security anomalies described in natural language, or any condition that requires interpretation. The LLM-driven checker handles these cases, with a confidence gate to prevent the model's inherent uncertainty from causing false alarms.
- **Thresholds should be calibrated from historical data, not set by intuition**: Standard deviation analysis against a historical baseline provides a principled foundation, with the expected false positive rate as the key tuning parameter.